In [ ]:
from pathlib import Path
import json
import numpy as np
from skimage.io import imsave
from nd2 import ND2File
import warnings
import pandas as pd
from sklearn.cluster import DBSCAN
from skimage.draw import ellipse_perimeter
from skimage.exposure import rescale_intensity
from skimage.color import gray2rgb
from matplotlib import pyplot as plt
from matplotlib import colors as mcolors
from scipy.optimize import OptimizeWarning

from calmutils.localization import refine_point_lsq
from calmutils.localization.util import sigma_to_full_width_at_quantile, full_width_at_quantile_to_sigma

def load_channels_from_nd2(file_path, channels):

    res = {}

    with ND2File(file_path) as reader:

        # nice OC name without whitespace
        channel_names = list(map(lambda s: s.channel.name.strip().replace(' ', '-'), reader.metadata.channels))
            
        for channel in channels:

            # try to find specified channel, otherwise warn and list available channels
            try:
                channel_idx = channel_names.index(channel)
            except ValueError:
                warnings.warn(f'channel {channel} not found in file {file_path}. available channels: {channel_names}')
                continue
            
            img = np.array(reader.to_dask()[:,channel_idx])
            res[channel] = img
        # invert xyz voxel size to zyx to match img array
        pixel_size = reader.voxel_size()[::-1]
    
    return res, pixel_size


def get_pixel_size(file_path):
    with ND2File(file_path) as reader:
        # invert xyz voxel size to zyx to match img array
        pixel_size = reader.voxel_size()[::-1]
    return pixel_size


def imsave_nowarnings(file, img, **kwargs):
    # catch low contrast warning
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', UserWarning)
        imsave(file, img, **kwargs)

def filter_clustering(blobs, pixel_size, expected_size, cluster_reject_distance, cluster_reject_n_spots):
    # drop sigma columns produced by blob_log/blob_dog
    blobs_just_coords = blobs * pixel_size / expected_size
    # spots that receive class -1 in DBSCAN := not in cluster
    # NOTE: explicitly setting algorithm='kd_tree' was necessary to avoid issues in multithreaded processing of files
    single_spot_idx = DBSCAN(cluster_reject_distance, min_samples=cluster_reject_n_spots, algorithm='kd_tree').fit_predict(blobs_just_coords) == -1
    return blobs[single_spot_idx], single_spot_idx


def refine_points(image, points):
    """
    Parameters
    ----------
    image: ndarray image to refine points in
    points: (n, image.ndim) array of n candidate points to refine
    """

    points_refined = []
    sigmas_refined = []
    minmax_refined = []
    for blob in points:

        # do Gaussian fit, ignore warnings about failed optimization -> we will skip those blobs
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', (OptimizeWarning, RuntimeWarning))
            pos_refined, fit = refine_point_lsq(image, blob)
        
        # skip if fit not possible or negative sigma
        if fit is None:
            continue
        fit, _ = fit
        if np.any(fit[-3:] < 0) or np.any(np.isnan(fit)):
            continue

        points_refined.append(pos_refined)
        sigmas_refined.append(fit[-3:])
        minmax_refined.append(fit[:2])

    points_refined = np.array(points_refined)
    return points_refined, np.array(sigmas_refined), np.array(minmax_refined)

def get_spot_visualization_projection(img, blobs, sigmas):

    # color to plot ellipse in
    ellipse_color = np.array(mcolors.hex2color(mcolors.XKCD_COLORS['xkcd:sea green']))
    # how much to expand the ellipse with radius = full width at tenth maximum
    ellipse_expansion_factor = 4
    # ellipse line width
    ellipse_line_width = 3

    img_projected = img.max(axis=0)
    proj_rgb = gray2rgb(rescale_intensity(img_projected, in_range=tuple(np.quantile(img_projected, (0.02, 0.9999))), out_range='float32'))

    for blob, sigma in zip(blobs, sigmas):
        
        # get position and sigma of blob in yx
        yx = blob[1:].astype(int)
        sy_sx = sigma[1:]

        # to radius of ellipse (based on full width at tenth maximum times expansion factor)
        ry_rx = (sigma_to_full_width_at_quantile(sy_sx, 0.1) / 2 * ellipse_expansion_factor).astype(int)
        
        # line width: draw single pixel ellipse at radius + 0, +1, ...
        for i in range(ellipse_line_width):
            proj_rgb[tuple(ellipse_perimeter(*yx, *(ry_rx+i), shape=proj_rgb.shape))] = ellipse_color

    proj_rgb = (proj_rgb * 255).astype(np.uint8)

    return proj_rgb

def blobs_to_df(blobs_i, file_path, sigmas: dict, minmax):

    pixel_size = get_pixel_size(file_path)

    df = pd.DataFrame()
    for channel_name, blobs_ii in blobs_i.items():
        df_i = pd.DataFrame({
            'spot_idx': np.arange(len(blobs_ii), dtype=int),
            'image_file': file_path,
            'channel': channel_name,
            **dict(zip('zyx', blobs_ii.T)), 
            **dict(zip(['z_micron', 'y_micron', 'x_micron'], (blobs_ii * pixel_size).T))
            })
        
        sigmas_i = sigmas.get(channel_name, None)
        minmax_i = minmax.get(channel_name, None)
        
        if sigmas_i is not None:
            for col_name, vals in zip(['sigma_z', 'sigma_y', 'simga_x'], sigmas_i.T):
                df_i[col_name] = vals
        if sigmas_i is not None:
            for col_name, vals in zip(['sigma_z_micron', 'sigma_y_micron', 'simga_x_micron'], (sigmas_i * pixel_size).T):
                df_i[col_name] = vals
        if minmax_i is not None:
            for col_name, vals in zip(['gauss_fit_min', 'gauss_fit_height'], minmax_i.T):
                df_i[col_name] = vals

        df = df.append(df_i, ignore_index=True)
    
    return df

In [ ]:
# path containing files to visualize
in_path = Path('/data/agl_data/NanoFISH/Gabi/GS075_20230818_K562-EVI1-GFP_t(3-8)_EVI-CTRL/')

# default: put results in subdirectory called 'spot-detection'
out_path = in_path / 'spot-detection'

# which channels to include
channels_to_include = ['561-CSU-W1', '640-CSU-W1']

# LoG threshold for all channels
thresholds_dog = 50.0

# Alternative: threshold_log can be a dictionary containing a separate threshold for each channel
thresholds_dog = {
    '561-CSU-W1': 70.0,
    '640-CSU-W1': 160.0
}

# make dict with same threshold for all channels
if not isinstance(thresholds_dog, dict):
    thresholds_dog = {channel: thresholds_dog for channel in channels_to_include}

# refine points via Gaussian fit?
do_gaussian_fit = True

# expected size (zyx, in microns)
expected_size = np.array([0.5, 0.25, 0.25])

# cluster rejection via DBSCAN clustering
# spots that lie in clusters in which at least cluster_reject_n_spots spots lie within cluster_reject_distance are ignored
# cluster_reject_distance is in units of expected size (i.e. 5.0: are considered to cluster if their distance is < 5*expected_size)
cluster_reject_distance = 5.0
cluster_reject_n_spots = 5

save_visualization = True

parameter_log = {
    'in_path': str(in_path),
    'channels_to_include': channels_to_include,
    'thresholds_dog': thresholds_dog,
    'do_gaussian_fit': do_gaussian_fit,
    'expected_size': list(expected_size),
    'cluster_reject_distance': cluster_reject_distance,
    'cluster_reject_n_spots': cluster_reject_n_spots
}


In [ ]:
# get all nd2 files in in_path
in_files = sorted(list(Path(in_path).glob('*.nd2')))

# show for verification
in_files

In [ ]:
from skimage.feature import blob_dog
from concurrent.futures import ThreadPoolExecutor

if not out_path.exists():
    out_path.mkdir(parents=True)

visualization_path = out_path / 'quick_result_visualization'
if save_visualization and not visualization_path.exists():
    visualization_path.mkdir(parents=True)


def load_and_detect_spots(in_file, channels_to_include, expected_size, threshold_dog,
                          do_visualization, cluster_reject_distance, cluster_reject_n_spots, do_refinement):

    images, pixel_size = load_channels_from_nd2(in_file, channels_to_include)

    # sigma for expected size in pixel
    sigma_expected = full_width_at_quantile_to_sigma(expected_size) / pixel_size

    blobs = {}
    sigmas = {}
    minmax = {}
    projections = {}
    for channel_name, img in images.items():

        # blob_log with single sigma at expected size
        # NOTE: img is converted to float to avoid internal rescaling of intensities
        blobs_i = blob_dog(img.astype(float), min_sigma=sigma_expected, max_sigma=sigma_expected, threshold=threshold_dog[channel_name])
        blobs_i, sigmas_i = blobs_i[:,:img.ndim], blobs_i[:,img.ndim:]

        if do_refinement:
            blobs_i, sigmas_i, minmax_i = refine_points(img, blobs_i)

        if len(blobs_i) > cluster_reject_n_spots:
            blobs_i, single_spot_idx = filter_clustering(blobs_i, pixel_size, expected_size, cluster_reject_distance, cluster_reject_n_spots)
            sigmas_i = sigmas_i[single_spot_idx]
            minmax_i = minmax_i[single_spot_idx]
            # print(len(blobs_i))
        
        blobs[channel_name] = blobs_i
        if do_refinement:
            sigmas[channel_name] = sigmas_i
            minmax[channel_name] = minmax_i

        if do_visualization:
            visualization_projection = get_spot_visualization_projection(img, blobs_i, sigmas_i)
            projections[channel_name] = visualization_projection

    return blobs, projections, sigmas, minmax

# do detection multithreaded
with ThreadPoolExecutor() as tpe:

    futures = [tpe.submit(load_and_detect_spots, 
                          in_file, channels_to_include, expected_size, thresholds_dog, save_visualization,
                           cluster_reject_distance, cluster_reject_n_spots, do_gaussian_fit) 
                for in_file in in_files]
        
    blobs = []
    visualization_projections = []
    sigmas = []
    minmaxs = []
    for in_file, f in zip(in_files, futures):
        blobs_i, visualization_projections_i, sigmas_i, minmax_i = f.result()
        for channel_name, b in blobs_i.items():
            print(f'{in_file}: number of detected blobs in {channel_name}: {len(b)}')
        blobs.append(blobs_i)
        visualization_projections.append(visualization_projections_i)
        sigmas.append(sigmas_i)
        minmaxs.append(minmax_i)

with open(out_path / 'spot_detection_parameters.json', 'w') as fd:
    json.dump(parameter_log, fd, indent=1) 

for i, in_file in enumerate(in_files):

    df = blobs_to_df(blobs[i], in_file, sigmas[i], minmaxs[i])
    out_file = out_path / (in_file.stem + '_spot-detection.csv')
    df.to_csv(out_file, index=None)

    if save_visualization:
        for channel_name, visualization_projection in visualization_projections[i].items():
            # make filepath for output
            outfile_visualization = visualization_path / (in_file.stem + f'_{channel_name}_spot-detection.png')            
            imsave_nowarnings(str(outfile_visualization), visualization_projection)

        

In [ ]:
from glob import glob

for f in glob('/data/agl_data/AndreasMaiser/NSD/23AM08-01/spot-detection/*.csv'):
    df = pd.read_csv(f)
    blobs = df[list('zyx') + ['z_micron']].values
    if len(blobs)>cluster_reject_n_spots:
        
# blobs

# from matplotlib import pyplot as plt

# plt.scatter(*blobs.T[1:])

        blobs_i = filter_clustering(blobs, get_pixel_size('/data/agl_data/AndreasMaiser/NSD/23AM08-01/23AM08-01_3_001.nd2'), expected_size, cluster_reject_distance, cluster_reject_n_spots)
# plt.scatter(*blobs_i.T[1:])